In [ ]:
import os
import torch
from datasets import load_dataset
from PIL import Image
from torch.utils.data import Dataset
from transformers import (
    AutoProcessor,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Configuration
os.environ["HUGGINGFACE_HUB_TOKEN"] = "<TOKEN>"
HF_TOKEN = os.getenv("HUGGINGFACE_HUB_TOKEN")
assert HF_TOKEN is not None, "Token not set."

ROOT = "DatasetFinal"
TRAIN_JSONL = "DatasetFinal/splits/stage1_visual_rcd/train.jsonl"
VAL_JSONL = "DatasetFinal/splits/stage1_visual_rcd/val.jsonl"
TEST_JSONL = "DatasetFinal/splits/stage1_visual_rcd/test.jsonl"
MODEL_NAME = "google/gemma-3-4b-it"
CHECKPOINT_DIR = "stage1_vision_checkpoints"
FINAL_ADAPTER_DIR = "stage1_vision_lora_adapter"

print(f"Stage 1: Visual Understanding Training")
print(f"Model: {MODEL_NAME}")
print(f"Final adapter: {FINAL_ADAPTER_DIR}/")



In [ ]:
train_ds = load_dataset("json", data_files=TRAIN_JSONL)["train"]
val_ds = load_dataset("json", data_files=VAL_JSONL)["train"]
test_ds = load_dataset("json", data_files=TEST_JSONL)["train"]

print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}")



In [ ]:
processor = AutoProcessor.from_pretrained(
    MODEL_NAME, token=HF_TOKEN, trust_remote_code=True
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)
print("Model loaded with 4-bit quantization")



In [ ]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=48,
    target_modules=r".*vision_tower.*self_attn\.(q_proj|k_proj|v_proj|out_proj)",
    modules_to_save=None,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
print("LoRA applied (r=32, alpha=48)")



In [ ]:
for name, param in model.named_parameters():
    if "language_model" in name or "lm_head" in name:
        param.requires_grad = False

projector_found = False
for name, param in model.named_parameters():
    if "multi_modal_projector" in name or "mm_projector" in name:
        param.requires_grad = True
        projector_found = True

if not projector_found:
    print("Warning: No projector found")

model.enable_input_require_grads()
model.config.use_cache = False
model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,}/{total:,} ({100*trainable/total:.2f}%)")



In [ ]:
class ChessVisionDataset(Dataset):
    def __init__(self, hf_dataset, root_folder, processor):
        self.ds = hf_dataset
        self.root = root_folder
        self.processor = processor
        self.tokenizer = processor.tokenizer

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        user_msg = item["messages"][0]["content"]
        assistant_msg = item["messages"][1]["content"]

        img_rel = user_msg[0]["image"]
        question_text = user_msg[1]["text"]
        answer_text = assistant_msg[0]["text"]

        image = Image.open(os.path.join(self.root, img_rel)).convert("RGB")

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": question_text.strip()},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": answer_text.strip()}],
            },
        ]

        chat_text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )

        enc = self.processor(text=chat_text, images=image, return_tensors="pt")

        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        pixel_values = enc["pixel_values"].squeeze(0)
        labels = input_ids.clone()

        # Mask user prompt
        assistant_start = chat_text.rfind("<assistant>")
        if assistant_start != -1:
            prefix = chat_text[:assistant_start]
            prefix_ids = self.tokenizer(prefix, add_special_tokens=False)["input_ids"]
            labels[: len(prefix_ids)] = -100

        # Mask image tokens
        labels[labels == 262144] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "pixel_values": pixel_values,
            "labels": labels,
        }


def collate_fn(batch):
    pixel_values = torch.stack([b["pixel_values"] for b in batch])
    input_ids = [b["input_ids"] for b in batch]
    attention_mask = [b["attention_mask"] for b in batch]
    labels = [b["labels"] for b in batch]

    padded = processor.tokenizer.pad(
        {"input_ids": input_ids, "attention_mask": attention_mask},
        padding=True,
        return_tensors="pt",
    )

    max_len = padded["input_ids"].shape[1]
    labels_padded = torch.full((len(labels), max_len), fill_value=-100, dtype=torch.long)

    for i, l in enumerate(labels):
        labels_padded[i, : l.shape[0]] = l

    return {
        "input_ids": padded["input_ids"],
        "attention_mask": padded["attention_mask"],
        "pixel_values": pixel_values,
        "labels": labels_padded,
    }


train_dataset = ChessVisionDataset(train_ds, ROOT, processor)
val_dataset = ChessVisionDataset(val_ds, ROOT, processor)
print(f"Datasets ready: Train={len(train_dataset):,}, Val={len(val_dataset):,}")



In [ ]:
training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=300,
    weight_decay=0.01,
    fp16=True,
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=2000,
    save_total_limit=3,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    report_to="tensorboard",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

print(f"Config: epochs={training_args.num_train_epochs}, "
      f"bs={training_args.per_device_train_batch_size}, "
      f"lr={training_args.learning_rate}, "
      f"schedule={training_args.lr_scheduler_type}")



In [ ]:
val_dataset_subset = val_dataset.ds.select(range(min(2000, len(val_dataset))))
val_dataset_small = ChessVisionDataset(val_dataset_subset, ROOT, processor)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset_small,
    tokenizer=processor.tokenizer,
    data_collator=collate_fn,
)

steps_per_epoch = len(train_dataset) // (
    training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
)
total_steps = steps_per_epoch * training_args.num_train_epochs
print(f"Trainer ready: {total_steps} total steps, {steps_per_epoch} steps/epoch")
print(f"Val subset: {len(val_dataset_small):,} samples")



In [ ]:
trainer.train()




In [ ]:
model.save_pretrained(FINAL_ADAPTER_DIR)
processor.save_pretrained(FINAL_ADAPTER_DIR) # Tokenizer config 
print(f"Saved to {FINAL_ADAPTER_DIR}/")
